# Kalshi 15-minute strategy experiment

Executed companion analysis for the isolated paper ledgers and strategy-lab wallets. Results are descriptive, fee-aware, and do not treat hindsight quotes as achievable fills.

## TL;DR

- The final main paper wallet closed 7 trades at 57.1% win rate and $-0.36 net P&L.
- Across 18 matched lab markets, adaptive exit captured 87.0% of winning-market hold upside; fixed exit captured 39.6%.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'research' / 'final_experiment_analysis.py').exists():
    raise RuntimeError('Execute this notebook from the repository root')
sys.path.insert(0, str(ROOT))
from research.final_experiment_analysis import (
    ExperimentRun, analyze_experiment
)
RUN_SPECS = [('throughput_50', 'data/simulations/20260825_throughput_50_2h'), ('prior_10', 'data/simulations/20260824_2h_consensus_reversal_counterfactual')]
runs = [ExperimentRun(label, ROOT / directory) for label, directory in RUN_SPECS]
analysis = analyze_experiment(runs, ROOT / 'models/commodity_directional.json')
saved = json.loads((ROOT / 'reports/20260825_throughput_50/analysis_summary.json').read_text())
assert analysis['aggregate'] == saved['aggregate']
print('Independent recomputation matches saved aggregate metrics.')

Independent recomputation matches saved aggregate metrics.


## Context and methods

Main-paper trades are assigned to a run by its recorded UTC start/end bounds. Strategy-lab wallets are isolated SQLite files. Fixed and adaptive exits are compared only when a hold-to-settlement wallet entered the same ticker and side in the same session. Winning-market upside capture and losing-market loss avoidance are reported separately.

In [2]:
run_rows = []
for run in analysis['runs']:
    run_rows.append({
        'run': run['label'], 'status': run['run']['status'],
        'snapshots': run['market']['market_snapshots'],
        'depth_rows': run['market']['orderbook_levels'],
        'main_trades': run['main']['closed'],
        'main_win_rate': run['main']['win_rate'],
        'main_pnl': run['main']['net_pnl'],
        'quality_passed': run['quality']['passed'],
        'model_hash': run['run']['model_hash'],
        'config_hash': run['run']['config_hash'],
    })
pd.DataFrame(run_rows)

,run,status,snapshots,depth_rows,main_trades,main_win_rate,main_pnl,quality_passed,model_hash,config_hash
0,throughput_50,COMPLETED,8389,299481,1,1.000000,32.2100,True,bundle:5036382a936772fc01e50e9a36a2fd886ffe91e...,744c7177ec96f1099290772848f921b3cfd3ba46e7c1d5...
1,prior_10,COMPLETED,8388,304006,7,0.571429,-0.3641,True,bundle:5036382a936772fc01e50e9a36a2fd886ffe91e...,93caad41e57a933b78e910c80aa0b08895c9f7e2360f9c...


## Decision funnel and refusal metadata

In [3]:
gate_rows = []
for run in analysis['runs']:
    total = run['decisions']['evaluations']
    for gate, count in run['decisions'].get('gate_counts', {}).items():
        gate_rows.append({
            'run': run['label'], 'gate': gate, 'evaluations': count,
            'share': count / total if total else None,
            'candidate_yes': run['decisions'].get('candidate_outcomes', {}).get('yes', 0),
            'candidate_no': run['decisions'].get('candidate_outcomes', {}).get('no', 0),
        })
pd.DataFrame(gate_rows).sort_values(['run', 'evaluations'], ascending=[True, False])

,run,gate,evaluations,share,candidate_yes,candidate_no
12,prior_10,entry_window,6660,0.793991,614,835
13,prior_10,min_probability,894,0.106581,614,835
14,prior_10,model_market_divergence,504,0.060086,614,835
15,prior_10,minimum_time_to_close,188,0.022413,614,835
16,prior_10,candle_freshness,74,0.008822,614,835
17,prior_10,min_edge,31,0.003696,614,835
18,prior_10,settlement_margin,12,0.001431,614,835
19,prior_10,entry_history,12,0.001431,614,835
20,prior_10,live_edge,6,0.000715,614,835
21,prior_10,cooldown,5,0.000596,614,835


## Strategy results

In [4]:
strategy = pd.DataFrame(analysis['aggregate']['strategy_lab'])
strategy.sort_values(['net_pnl', 'closed'], ascending=[False, False])

,strategy,runs_present,closed,wins,losses,win_rate,net_pnl,pnl_per_closed_position
7,early_consensus_240,1,3,3,0,1.000000,68.6903,22.896767
8,early_consensus_300,1,3,3,0,1.000000,68.6903,22.896767
9,early_consensus_360,1,3,3,0,1.000000,68.6903,22.896767
6,early_consensus_180,1,2,2,0,1.000000,23.2832,11.641600
12,microprice_flow,2,40,14,26,0.350000,16.8457,0.421142
1,change_point_flip,2,15,8,7,0.533333,3.2931,0.219540
2,complementary_pair_arb,2,0,0,0,NaN,0.0000,NaN
18,trend_trailing,2,0,0,0,NaN,0.0000,NaN
16,regime_ensemble,2,35,18,17,0.514286,-15.8287,-0.452249
4,consensus_tail_carry,2,8,7,1,0.875000,-18.7942,-2.349275


## Matched exit-policy comparison

In [5]:
exit_rows = []
for run in analysis['runs']:
    for row in run['matched_exit_comparison']:
        exit_rows.append({'run': run['label'], **row})
pd.DataFrame(exit_rows)

,run,strategy,matched_positions,actual_pnl,hold_counterfactual_pnl,delta_vs_hold,winner_markets,winner_upside_capture,winner_pnl,winner_hold_pnl,loser_markets,loser_exit_pnl,loser_hold_pnl,loss_avoided
0,throughput_50,fixed_take_profit,13,-51.5613,-58.1396,6.5783,6,0.380368,50.9376,133.9165,7,-102.4989,-192.0561,89.5572
1,throughput_50,adaptive_take_profit,11,-23.0172,11.0146,-34.0318,6,0.745873,99.8847,133.9165,5,-122.9019,-122.9019,0.0000
2,prior_10,fixed_take_profit,22,-42.1656,-11.4005,-30.7651,13,0.396307,27.3867,69.1047,9,-69.5523,-80.5052,10.9529
3,prior_10,adaptive_take_profit,18,-37.5440,-31.1039,-6.4401,9,0.869637,42.9612,49.4013,9,-80.5052,-80.5052,0.0000


## Main-paper trade decisions

In [6]:
detail = []
for run in analysis['runs']:
    detail.extend({'run': run['label'], **row} for row in run['main_trade_details'])
pd.DataFrame(detail)

,run,trade_id,asset,ticker,outcome,status,exchange_status,placed_at,exited_at,contracts,entry_price,exit_price,entry_probability,pnl,maximum_pnl_if_winner,positive_exit_capture_ratio
0,throughput_50,6a5d236a-9b08-4c83-97d2-42ecaa7970de,GOLD,KXGOLD15M-26AUG250330-30,no,WON,finalized,2026-08-25T07:27:38.451774+00:00,NaN,81.95,0.59,NaN,0.679568,32.2100,32.21,1.000000
1,prior_10,2c4f828c-810b-48a3-9e7e-1f36dd797524,BTC,KXBTC15M-26AUG242100-00,yes,EXITED,take_profit,2026-08-25T00:46:08.465462+00:00,2026-08-25T00:52:18.485531+00:00,12.00,0.77,0.955,0.830032,2.0328,2.61,0.778851
2,prior_10,56491aa0-6a5a-4cca-80b7-c0ac9ee1d48b,BTC,KXBTC15M-26AUG242115-15,yes,EXITED,confirmed_reversal,2026-08-25T01:03:28.505929+00:00,2026-08-25T01:10:38.459743+00:00,13.00,0.71,0.220,0.759106,-6.7173,3.58,NaN
3,prior_10,68750ff9-a96f-44b8-9552-11b1cd764f94,BTC,KXBTC15M-26AUG242130-30,yes,WON,finalized,2026-08-25T01:21:08.431001+00:00,NaN,13.00,0.72,NaN,0.759936,3.4500,3.45,1.000000
4,prior_10,f14fc85a-be18-4985-9d77-ea3222ec4bd6,BTC,KXBTC15M-26AUG242145-45,no,EXITED,confirmed_reversal,2026-08-25T01:33:18.466822+00:00,2026-08-25T01:38:48.455352+00:00,13.00,0.72,0.550,0.769824,-2.6262,3.45,NaN
5,prior_10,4168b74e-d480-4df1-83fb-ad940dcf41ef,BTC,KXBTC15M-26AUG242200-00,no,EXITED,confirmed_reversal,2026-08-25T01:52:58.526784+00:00,2026-08-25T01:58:18.327895+00:00,11.00,0.83,0.270,0.866909,-6.4218,1.76,NaN
6,prior_10,0a5d94eb-5f88-4e55-8b33-72e72237da0e,SILVER,KXSILVER15M-26AUG242215-15,no,EXITED,take_profit,2026-08-25T02:03:58.517904+00:00,2026-08-25T02:05:28.581313+00:00,12.00,0.80,0.982,0.849286,2.0284,2.26,0.897522
7,prior_10,f2380e5e-433e-4791-b129-c465d508af11,SOL,KXSOL15M-26AUG242215-15,no,WON,finalized,2026-08-25T02:12:58.808363+00:00,NaN,17.83,0.54,NaN,0.622106,7.8900,7.89,1.000000


## Model diagnostics

In [7]:
models = pd.DataFrame(analysis['models'])
models.sort_values('auc', ascending=False)

,asset,auc,baseline_auc,indicator_delta_auc,deployment_status,weight
3,HYPE,0.559809,0.513889,0.045920,active,0.089714
6,SOL,0.532485,0.520224,0.012262,shadow_only,0.000000
5,SILVER,0.526142,0.521152,0.004991,shadow_only,0.000000
1,DOGE,0.523349,0.491560,0.031789,shadow_only,0.000000
7,WTI,0.520297,0.493425,0.026872,shadow_only,0.000000
2,GOLD,0.512484,0.550796,-0.038312,shadow_only,0.000000
9,ZEC,0.508488,0.521338,-0.012850,shadow_only,0.000000
4,NEAR,0.507041,0.492761,0.014280,shadow_only,0.000000
8,XRP,0.497939,0.487770,0.010169,shadow_only,0.000000
0,BNB,0.464392,0.435444,0.028949,shadow_only,0.000000


## Limitations and takeaways

- The runs cover different market windows and configuration hashes; cross-run differences are descriptive, not causal. The same-window strategy-lab comparison is the strongest available evidence.
- A high win rate can coexist with negative P&L when losses dominate many small wins. Promotion therefore requires positive fee-adjusted expectancy, adequate sample size, and stable results across independent windows.
- Model AUCs near 0.50 imply weak ranking power. Only artifacts that cleared the deployment threshold were allowed a non-zero blend weight.
- No result guarantees live profitability; fill latency, queue position, and partial fills remain material execution risks.